In [5]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
df = pd.read_excel("input.xlsx")
print(df.columns.tolist())

['Application Name', 'District Name', 'Consumer Number', 'PV Capacity (kW)', 'Discom Name', 'Mobile Number', 'Consumer Address']


Done!


In [3]:
import pandas as pd
import re

# Read file
df = pd.read_excel("input.xlsx")

# Clean column names
df.columns = df.columns.str.strip()

# Create output dataframe
new_df = pd.DataFrame()

# Customer Name in Proper Case
new_df["Customer Name"] = (
    df["Application Name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.title()
)

# Contact Number
new_df["Contact Number"] = (
    df["Mobile Number"]
    .fillna("Not Available")
)

new_df["Contact Number"] = (
    new_df["Contact Number"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

new_df.loc[
    (new_df["Contact Number"] == "") |
    (new_df["Contact Number"].str.lower() == "nan"),
    "Contact Number"
] = "Not Available"

# Complete Address
new_df["Complete Address"] = (
    df["Consumer Address"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Extract PIN Code
def extract_pincode(address):
    match = re.search(r"\b\d{6}\b", str(address))
    return match.group(0) if match else ""

new_df["Pin Code"] = new_df["Complete Address"].apply(extract_pincode)

# City
def extract_city(address, district):
    address = str(address).lower()

    city_keywords = [
        "prayagraj",
        "allahabad",
        "handia",
        "jhunsi",
        "naini",
        "soraon",
        "phulpur",
        "koraon",
        "meja"
    ]

    for city in city_keywords:
        if city in address:
            return city.title()

    return str(district).title()

new_df["City"] = df.apply(
    lambda row: extract_city(
        row["Consumer Address"],
        row["District Name"]
    ),
    axis=1
)

# State
new_df["State"] = "Uttar Pradesh"

# Remaining fields
new_df["PV Capacity (kW)"] = df["PV Capacity (kW)"]

new_df["Discom Name"] = (
    df["Discom Name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

new_df["Connection Number"] = (
    df["Consumer Number"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
)

# Save CSV
new_df.to_csv(
    "PrayagLeads.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV created successfully!")
print(f"Total records: {len(new_df)}")

CSV created successfully!
Total records: 966
